In [1]:
#importer les packages
import os
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()

from utils.correcting_features import corrige_all, corrige_etabli2, corrige_rgp2, corrige_rgp3, corrige_op_ing
from utils.load_corrections import get_all_correctifs_from_google, load_all_correctifs
from utils.traitements import importtab, gentab

In [2]:
# ATTENTION, avant de commencer, s'assurer d'avoir un dossier "INPUT" 
# avec les données au format parquet pour chaque année et un dossier
# POST_GENTAB pour acceuillir les données traitées (attention c'est assez volumineux)

# Modifier la page format du googlesheet si il y a de nouveaux formats cette année
# Modifier également la page D_EPE du googlesheet ESR pour ajouter les EPE de la nouvelle année

In [3]:
DATA_PATH=os.getcwd()

In [4]:
url_atlas='https://docs.google.com/spreadsheet/ccc?key=11NFXSIg6gQMCsMa8zWQQyypvvYBEmfyJfF2yytXqgMk&output=xls'
url_esr='https://docs.google.com/spreadsheet/ccc?key=1FwPq5Qw7Gbgj_sBD6Za4dfDDk6ydozQ99TyRjLkW5d8&output=xls'

VARS_atlas=['COM_U','COM_M', 'ETABLI', 'MINISTER', 'DC', 'NATION', 'ETABLI2', 'FORMAT','CURSUS_LMD', 'DISCIPLI','RGP2', 'RGP3','OP_ING','SECRET']
VARS_esr=['O_DUTBUT','J_LMDDONT','DISCIPLINES_SISE','FORMAT','D_EPE','C_ETABLISSEMENTS','LES_COMMUNES_26_voir grist']

Charger les données des googlesheets depuis google en local au format .json dans le dossier (pas besoin de le faire à chaque fois, à faire si c'est la première fois OU lors d'une modification sur le.s googlesheet.s)

In [5]:
#get_all_correctifs_from_google(url_atlas,VARS_atlas,'_atlas')
#get_all_correctifs_from_google(url_esr,VARS_esr,'_esr')

Charger les 2 googlessheets dans le notebook

In [6]:
CORRECTIFS_dict_esr = load_all_correctifs('_esr')
CORRECTIFS_dict = load_all_correctifs('_atlas')

In [7]:
formats=pd.DataFrame(CORRECTIFS_dict_esr['FORMAT']).drop_duplicates()

On charge bien la rentrée scolaire de l'année en cours

In [ ]:
rentree_sco=2024

On charge les données correspondantes

In [9]:
df=pd.read_parquet(f"./INPUT/ts{str(rentree_sco)[2:]}d_tot.parquet", engine='fastparquet')

In [10]:
df.columns=[x.upper() for x in df.columns]

In [11]:
df[pd.isna(df.FORMAT)]
#df.loc[(pd.isna(df.FORMAT))&(df.ETABLI=='9830445S'),'FORMAT'] = 'autr'

,SIGLE_U,SECT_U,NAT_U,COM_U,LIB1_U,LIB2_U,ACA_U,SIGLE_M,SECT_M,NAT_M,...,NFIO,NFIINP,FINECOLE,DISC,DC,SAN,SANN,SOC,SOCN,SECT


In [12]:
df[pd.isna(df.ETABLI)]

,SIGLE_U,SECT_U,NAT_U,COM_U,LIB1_U,LIB2_U,ACA_U,SIGLE_M,SECT_M,NAT_M,...,NFIO,NFIINP,FINECOLE,DISC,DC,SAN,SANN,SOC,SOCN,SECT


In [13]:
df[pd.isna(df.COMPOS)]

,SIGLE_U,SECT_U,NAT_U,COM_U,LIB1_U,LIB2_U,ACA_U,SIGLE_M,SECT_M,NAT_M,...,NFIO,NFIINP,FINECOLE,DISC,DC,SAN,SANN,SOC,SOCN,SECT


In [14]:
df[pd.isna(df.SEXE)]

,SIGLE_U,SECT_U,NAT_U,COM_U,LIB1_U,LIB2_U,ACA_U,SIGLE_M,SECT_M,NAT_M,...,NFIO,NFIINP,FINECOLE,DISC,DC,SAN,SANN,SOC,SOCN,SECT


In [15]:
#verifier que HCPGE est bien rempli
if 'HCPGE' in list(df.columns):
    print(df.HCPGE.value_counts())

In [16]:
#voir les effectifs sans les doubles comptes
if 'HSIFA' in list(df.columns):
    print(df.EFFTOT.sum()-df.loc[(df['HSIFA']==0.0),'EFFTOT'].sum())
    print(df.loc[(df['HSIFA']==0.0),'EFFTOT'].sum())

On applique la fonction importtab, qui va corriger les features une par une de cette année

In [17]:
df_up=importtab(df,CORRECTIFS_dict,CORRECTIFS_dict_esr,corrige_etabli2,corrige_all,rentree_sco)

c:\Users\haallat\Documents\atlas-opendata\utils\traitements.py:51: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'DU' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[df['TYP_DIPL'].isin(["01","02","04","05","06","07","08","09","17","UA","UB","UC","UD","UE","UF","UG","UI","UJ","UO","UR","US","UT","UU","UV","UY","XF"]),'DNDU']='DU'


COM_U
COM_M
ETABLI
DISCIPLI
FORMAT
MINISTER
NATION
CURSUS_LMD


c:\Users\haallat\Documents\atlas-opendata\utils\correcting_features.py:184: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'L' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[((df[VAR] == '')|(pd.isna(df[VAR])) |(df[VAR]==None)) & (df['ETABLI']==c['ETABLI']) & (df['RENTREE']==c['ANNEE']), VAR]=c['OUT']


ETABLI2


In [18]:
# test EFFTOT si ils sont toujours egaux sans HSIFA
if 'HSIFA' in list(df.columns):
    print(df_up['HSIFA'].value_counts())
    print(df_up.EFFTOT.sum()-df_up.loc[(df_up['HSIFA']==0.0),'EFFTOT'].sum()==df.EFFTOT.sum()-df.loc[(df['HSIFA']==0.0),'EFFTOT'].sum())

Series([], Name: count, dtype: int64)
True


In [19]:
# test si nouveaux formats cette année
[x for x in list(df_up.FORMAT.drop_duplicates()) if x not in list(pd.DataFrame(CORRECTIFS_dict_esr['FORMAT'])['FORMAT'].drop_duplicates())]

[]

In [20]:
df_up.loc[df_up.FORMAT=='']

,SIGLE_U,NAT_U,COM_U,LIB1_U,LIB2_U,SIGLE_M,COM_M,LIB1_M,LIB2_M,ETABLI,...,ETABLISSEMENT,UNIV,HSIFA,EFFECTIF_SANS_DOUBLE_COMPTE,HCPGE,SPECIUT,LMDDONT,LMDDONTBIS,DISCIPLI,ETABLI2


In [21]:
df_up.loc[df_up.FORMAT=='None']

,SIGLE_U,NAT_U,COM_U,LIB1_U,LIB2_U,SIGLE_M,COM_M,LIB1_M,LIB2_M,ETABLI,...,ETABLISSEMENT,UNIV,HSIFA,EFFECTIF_SANS_DOUBLE_COMPTE,HCPGE,SPECIUT,LMDDONT,LMDDONTBIS,DISCIPLI,ETABLI2


In [22]:
#fonction gentab

In [23]:
df_up2= gentab(df_up, rentree_sco, CORRECTIFS_dict, CORRECTIFS_dict_esr, corrige_rgp2, corrige_rgp3, corrige_op_ing)

RGP2
RGP3
OP_ING


On regarde si les effectifs des apprentis sont présents

In [24]:
if 'EFF_STS_APP' in df_up2.columns:
    print(df_up2.EFF_STS_APP.value_counts())

EFF_STS_APP
0    26169
Name: count, dtype: Int64


Pour chaque année, on enregistre les données traitées

In [25]:
df_up2.to_json(f"./POST_GENTAB/atlas{str(rentree_sco)}.json")

In [26]:
df_up2

,RENTREE,ENQ,MINISTER,SECT,COMPOS,NAT_U,SIGLE_U,LIB1_U,LIB2_U,COM_CODE,...,FILIERE,ETABLISSEMENT,UNIV,SEXE,memeCOM,memeUUCR,EFFTOT,EFFSDC,EFF_STS_APP,ID
0,2001,agri,03,PR,0021818R,301,LPA,LYCEE ROBERT SCHUMAN,EPICEA,02173,...,nan,nan,nan,1,True,True,29,0,0,R32
1,2001,agri,03,PR,0021818R,301,LPA,LYCEE ROBERT SCHUMAN,EPICEA,02173,...,nan,nan,nan,2,True,True,6,0,0,R32
2,2001,agri,03,PR,0030888Z,320,ECA ..,ECOLE AGRICOLE PRIVEE,CLAUDE MERCIER,03165,...,nan,nan,nan,1,True,True,18,0,0,R84
3,2001,agri,03,PR,0030888Z,320,ECA ..,ECOLE AGRICOLE PRIVEE,CLAUDE MERCIER,03165,...,nan,nan,nan,2,True,True,2,0,0,R84
4,2001,agri,03,PR,0030925P,320,ECA ..,ECOLE AGRICOLE PRIVEE,INSTITUT RURAL,03109,...,nan,nan,nan,1,True,True,7,0,0,R84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
26164,2001,the1,80,PU,0780015T,302,LMN,LYCEE MILITAIRE,MILITAIRE,78545,...,nan,nan,nan,2,True,True,9,0,0,R11
26165,2001,the1,80,PU,0780015T,302,LMN,LYCEE MILITAIRE,MILITAIRE,78545,...,nan,nan,nan,1,True,True,45,0,0,R11
26166,2001,the1,80,PU,0780015T,302,LMN,LYCEE MILITAIRE,MILITAIRE,78545,...,nan,nan,nan,2,True,True,20,0,0,R11
26167,2001,the1,80,PU,0780015T,302,LMN,LYCEE MILITAIRE,MILITAIRE,78545,...,nan,nan,nan,1,True,True,112,0,0,R11


In [27]:
df_up2[df_up2.COM_CODE=='nan'] #checker la variable finecole (finness) si c'est des codes postaux

,RENTREE,ENQ,MINISTER,SECT,COMPOS,NAT_U,SIGLE_U,LIB1_U,LIB2_U,COM_CODE,...,FILIERE,ETABLISSEMENT,UNIV,SEXE,memeCOM,memeUUCR,EFFTOT,EFFSDC,EFF_STS_APP,ID
